# 02 — Data Cleaning

**Package:** `fraud_nb01-07_v2` (flat S3 layout, shared verified helpers)

**Phase 1, notebook 2 of 7.** Reads `01_raw_loaded.parquet`, turns the raw landing extract into
typed, canonical values, joins the cleaned dimensions down to payment grain, and adds
deterministic row-wise features.

**Nothing in this notebook is fitted.** Every feature is row-wise or strictly backward-looking —
the last-login join and the velocity counts use only rows at or before each payment — so nothing
can leak across the split. Fitted transforms (imputation,
outlier bounds, scalers, encoders) start in notebook 03 and are fitted on `__split == 'train'`
only.

**Order matters: normalise sentinels → canonicalise → map → type-cast.** A `.map()` returns
`NaN` for every unmapped value, so mapping before sentinel normalisation converts `'UNKNOWN'`
into an unfillable null that `StandardScaler` will happily ignore and then propagate.

## 0. Colab bootstrap

Keys come from **Colab Secrets** (the key icon in the left sidebar), not from a cell. Add two
secrets and enable notebook access for both:

| Secret name | Value |
|---|---|
| `AWS_ACCESS_KEY_ID` | your IAM access key id |
| `AWS_SECRET_ACCESS_KEY` | your IAM secret |

They are loaded into environment variables so that **boto3 still resolves through the default
credential chain** — the client construction below is byte-identical to what runs in production
under IRSA. Passing keys directly into `boto3.client()` would disable IRSA later and is never
done here.

A hosted runtime is a credential-leak surface: anything pasted into a cell persists in the
notebook file and its autosave history. Nothing below prints a key.

In [ ]:
%pip install -q boto3==1.43.95

## 1. Configuration and S3 helpers

In [ ]:
# MARKER: fraud_nb01-07_v2 :: 02_Data_Cleaning
import io, os, json, time, hashlib, platform, importlib
from datetime import datetime, timezone
import boto3
from botocore.exceptions import ClientError
import joblib
import numpy as np
import pandas as pd
from google.colab import userdata

BUCKET, REGION = "fraud-ecommerce", "ap-south-2"
SPLIT_DATE = pd.Timestamp("2025-07-01")      # stamped in notebook 01 as __split; verified here, never recomputed
CONTRACT_VERSION = "v1"
SEED = 42
MARKER = "fraud_nb01-07_v2"
for _k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"):
    if not os.environ.get(_k):
        os.environ[_k] = userdata.get(_k)     # Colab Secrets -> process env only; never printed or saved
s3 = boto3.client("s3", region_name=REGION)
RAW, LABELS, CONTRACTS, DATA, REPORTS = "raw/", "raw/label_sources/", "contracts/", "data/", "reports/"  # flat layout
ident = boto3.client("sts", region_name=REGION).get_caller_identity()
print("account:", ident["Account"], "| arn:", ident["Arn"])
if ident["Arn"].endswith(":root"):
    print("NOTE: running as root — accepted for Phase 1; move to an IAM principal before Phase 2")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)


def _jsonable(o):
    if isinstance(o, (np.integer, np.floating, np.bool_)):
        return o.item()
    if isinstance(o, (pd.Timestamp, datetime)):
        return o.isoformat()
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(f"not JSON-serialisable: {type(o).__name__}")


def read_bytes_s3(key):
    return s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()


def put_bytes_s3(body, key):
    s3.put_object(Bucket=BUCKET, Key=key, Body=body)
    print(f"saved s3://{BUCKET}/{key}  ({len(body):,} bytes)")


def key_exists(key):
    try:
        s3.head_object(Bucket=BUCKET, Key=key)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] in ("404", "NoSuchKey", "NotFound"):
            return False
        raise


def read_s3(key):
    return pd.read_parquet(io.BytesIO(read_bytes_s3(key)))


def save_s3(df, key):
    """Parquet only; the bytes are verified to round-trip columns, dtypes and categories before upload."""
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    body = buf.getvalue()
    back = pd.read_parquet(io.BytesIO(body))
    assert list(back.columns) == list(df.columns) and len(back) == len(df), f"parquet round-trip changed shape: {key}"
    bad = [c for c in df.columns if str(back[c].dtype) != str(df[c].dtype)]
    assert not bad, f"parquet round-trip changed dtypes in {key}: {bad}"
    badcat = [c for c in df.columns if str(df[c].dtype) == "category"
              and list(back[c].cat.categories) != list(df[c].cat.categories)]
    assert not badcat, f"parquet round-trip changed categories in {key}: {badcat}"
    put_bytes_s3(body, key)


def read_json_s3(key):
    return json.loads(read_bytes_s3(key))


def save_json_s3(obj, key):
    put_bytes_s3(json.dumps(obj, indent=1, default=_jsonable).encode(), key)


def save_model_s3(obj, key):
    buf = io.BytesIO()
    joblib.dump(obj, buf)
    put_bytes_s3(buf.getvalue(), key)


def load_model_s3(key):
    return joblib.load(io.BytesIO(read_bytes_s3(key)))


# names used by notebooks 01-04 (same verified implementations underneath)
def s3_read_csv(key, **kw):
    return pd.read_csv(io.BytesIO(read_bytes_s3(key)), **kw)


s3_read_parquet, s3_read_json = read_s3, read_json_s3


def s3_write_parquet(df, key):
    save_s3(df, key)
    return f"s3://{BUCKET}/{key}"


def s3_write_json(obj, key):
    save_json_s3(obj, key)
    return f"s3://{BUCKET}/{key}"


RAW_KEYS = ([f"{RAW}{t}.csv" for t in ["payments", "orders", "order_items", "account_logins", "customers",
                                         "merchants", "cards", "devices", "ip_reputation"]]
            + [f"{LABELS}{t}.csv" for t in ["fraud_events", "audit_sample", "chargebacks"]]
            + [f"{CONTRACTS}schema_v1.json"])
_absent = [k for k in RAW_KEYS if not key_exists(k)]
assert not _absent, f"missing landing objects in s3://{BUCKET}/: {_absent}"
print(f"landing objects present: {len(RAW_KEYS)} (flat layout, bucket root)")


def run_meta(notebook):
    return {"notebook": notebook, "marker": MARKER,
            "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "library_versions": LIB_VERSIONS, "version_drift": VERSION_DRIFT}


LIBS = ["pandas", "numpy", "pyarrow", "scipy", "statsmodels", "sklearn", "lightgbm", "xgboost", "joblib", "boto3"]
LIB_VERSIONS = {"python": platform.python_version(),
                **{m: importlib.import_module(m).__version__ for m in LIBS}}
EXPECTED = {"pandas": "2.2.3", "numpy": "2.1.3", "pyarrow": "23.0.1", "scipy": "1.16.3", "statsmodels": "0.15.0",
            "sklearn": "1.6.1", "lightgbm": "4.6.0", "xgboost": "3.4.1", "boto3": "1.43.95"}
VERSION_DRIFT = {m: {"verified": v, "found": LIB_VERSIONS[m]} for m, v in EXPECTED.items() if LIB_VERSIONS[m] != v}
if not LIB_VERSIONS["python"].startswith("3.13."):
    VERSION_DRIFT["python"] = {"verified": "3.13.x", "found": LIB_VERSIONS["python"]}
print(LIB_VERSIONS)
print("VERSION DRIFT vs the runtime verified on 2026-09-16:", VERSION_DRIFT or "none")

## 2. Cleaning primitives

Four functions, used everywhere below. Defined once so notebook 06 and `src/preprocess.py` can
reuse the identical implementation — one transform implementation is what makes train/serve
parity provable rather than hoped for.

In [ ]:
SENTINELS = {'NA', 'N/A', 'NAN', 'NULL', 'NONE', 'UNKNOWN', 'XX', '-1', '999', '999999',
             '000000', 'AS0', 'OTHER', 'UNKNOWN BANK', 'NIL', '?', 'TBD'}

def sclean(s):
    """Strip padding, then convert blanks and sentinel tokens to a real null."""
    o = s.astype(str).str.strip()
    return o.mask(o.str.upper().isin(SENTINELS) | (o == '') | (o.str.lower() == 'nan'))

def parse_datetime(s):
    """Parse format by format. NEVER dayfirst globally: it silently turns ISO 2025-11-02
    into 11 February and gives you dates that are wrong rather than an error."""
    x = s.astype(str).str.strip().str.replace(r'\+05:30$', '', regex=True)
    out = pd.to_datetime(x, format='%Y-%m-%dT%H:%M:%S', errors='coerce')
    for fmt in ('%Y-%m-%d %H:%M:%S', '%d/%m/%Y %H:%M', '%Y-%m-%d'):
        m = out.isna()
        if not m.any():
            break
        out[m] = pd.to_datetime(x[m], format=fmt, errors='coerce')
    return out

def to_num(s):
    """Numeric, tolerating thousands separators written by an upstream export."""
    return pd.to_numeric(sclean(s).str.replace(',', '', regex=False), errors='coerce')

def to_bool(s):
    return sclean(s).str.upper().map({'1': 1, '0': 0, 'Y': 1, 'N': 0, 'TRUE': 1,
                                      'FALSE': 0, '1.0': 1, '0.0': 0}).astype('Float64')

# Title-casing is NOT safe as a canonicaliser: it turns UPI into 'Upi' and EMI into 'Emi'.
# Canonical labels come from an explicit upper-cased lookup; title-case is only the fallback.
def canonicalise(s, mapping=None):
    x = sclean(s).str.replace(r'\s+', ' ', regex=True).str.strip()
    fallback = x.str.title()
    if mapping:
        m = {k.upper(): v for k, v in mapping.items()}
        u = x.str.upper()
        return pd.Series(np.where(u.isin(m.keys()), u.map(m), fallback),
                         index=x.index).where(x.notna())
    return fallback.where(x.notna())

CANON = {
 'payment_method': {'CREDIT CARD': 'Credit Card', 'CREDIT_CARD': 'Credit Card',
   'CC': 'Credit Card', 'DEBIT CARD': 'Debit Card', 'DEBIT_CARD': 'Debit Card',
   'DC': 'Debit Card', 'COD': 'Cash on Delivery', 'CASH ON DELIVERY': 'Cash on Delivery',
   'NETBANKING': 'Net Banking', 'NET_BANKING': 'Net Banking', 'NET BANKING': 'Net Banking',
   'UPI': 'UPI', 'WALLET': 'Wallet', 'EMI': 'EMI'},
 'gateway': {'RAZORPAY': 'Razorpay', 'PAYU': 'PayU', 'CASHFREE': 'Cashfree',
   'BILLDESK': 'BillDesk', 'AMAZON PAY GATEWAY': 'Amazon Pay Gateway'},
 'ship_speed': {'EXPRESS': 'Express', 'EXP': 'Express', 'STANDARD': 'Standard',
   'STD': 'Standard', 'SAME DAY': 'Same Day', 'SAME_DAY': 'Same Day',
   'SAMEDAY': 'Same Day', 'SCHEDULED': 'Scheduled'},
 'city': {'BANGALORE': 'Bengaluru', 'BOMBAY': 'Mumbai', 'CALCUTTA': 'Kolkata',
   'MADRAS': 'Chennai'},
 'email': {'MAINSTREAM': 'mainstream', 'CORPORATE': 'corporate', 'NICHE': 'niche',
   'DISPOSABLE': 'disposable'},
}
CANON.update({
 'device_type': {'IPHONE': 'iPhone', 'ANDROID PHONE': 'Android Phone', 'WINDOWS DESKTOP': 'Windows Desktop',
                 'MAC': 'Mac', 'TABLET': 'Tablet'},
 'os_family': {'IOS': 'iOS', 'MACOS': 'macOS', 'ANDROID': 'Android', 'WINDOWS': 'Windows'},
 'browser_family': {'WEBVIEW': 'WebView', 'CHROME MOBILE': 'Chrome Mobile'},
 'network': {'RUPAY': 'RuPay', 'VISA': 'Visa', 'MASTERCARD': 'Mastercard', 'AMEX': 'Amex'},
})
# Title-case is only a fallback; brand spellings above are explicit (the same class of bug as UPI -> 'Upi').
EXPECTED_LEVELS = {
 'device_type': {'Android Phone', 'iPhone', 'Windows Desktop', 'Mac', 'Tablet'},
 'os_family': {'Android', 'iOS', 'Windows', 'macOS'},
 'browser_family': {'Chrome Mobile', 'Chrome', 'Safari', 'Edge', 'Firefox', 'WebView'},
 'network': {'Visa', 'Mastercard', 'RuPay', 'Amex'},
 'city': {'Mumbai', 'Hyderabad', 'Kochi', 'Kolkata', 'Pune', 'Chennai', 'Lucknow', 'Ahmedabad', 'Indore',
          'Surat', 'Chandigarh', 'Delhi', 'Bengaluru', 'Jaipur'},
 'email_domain_class': {'mainstream', 'disposable', 'corporate', 'niche'},
 'shipping_speed': {'Standard', 'Express', 'Scheduled', 'Same Day'},
}


def assert_levels(frame, col):
    got = set(frame[col].dropna().unique())
    extra = got - EXPECTED_LEVELS[col]
    assert not extra, f'{col}: unexpected canonical levels {sorted(extra)} — extend CANON, do not ignore'

def asof_distinct_count(frame, key, entity="customer_id", ts="payment_ts"):
    """Distinct `entity` values seen on `key` at or before each row's `ts` (inclusive). NaN where `key` is null.
    Point-in-time by construction: a row never sees a later first appearance, so no split boundary is needed."""
    f = frame.loc[frame[key].notna(), [key, entity, ts]]
    first = (f.groupby([key, entity], as_index=False, sort=True)[ts].min()
               .sort_values(ts, kind="mergesort"))
    first["_n"] = first.groupby(key, sort=False).cumcount() + 1
    rows = f.assign(_row=f.index).sort_values(ts, kind="mergesort")
    m = pd.merge_asof(rows, first[[key, ts, "_n"]], on=ts, by=key,
                      direction="backward", allow_exact_matches=True)
    assert m["_n"].notna().all(), f"as-of join left gaps for {key}"
    return m.set_index("_row")["_n"].reindex(frame.index).astype("float64")


VELOCITY = {"device_n_customers": "device_id", "ip_n_customers": "ip_address"}
print('primitives defined')

## 3. Clean the payment spine

In [ ]:
df = s3_read_parquet(f'{DATA}01_raw_loaded.parquet')
n_in = len(df)
print(f'loaded {n_in:,} x {df.shape[1]}')

for col in ['payment_amount', 'processing_fee', 'account_age_days_at_txn',
            'attempt_seq_in_session', 'ip_region_code']:
    df[col] = to_num(df[col])
for col in ['is_3ds_attempted', 'is_3ds_success', 'is_guest_checkout']:
    df[col] = to_bool(df[col])
df['payment_method']  = canonicalise(df.payment_method, CANON['payment_method'])
df['payment_gateway'] = canonicalise(df.payment_gateway, CANON['gateway'])
for col in ['ip_country', 'ip_asn', 'device_id', 'card_token', 'session_id', 'ip_address']:
    df[col] = sclean(df[col])

print('payment_method :', sorted(df.payment_method.dropna().unique()))
print('payment_gateway:', sorted(df.payment_gateway.dropna().unique()))
assert df.payment_method.nunique() == 7, 'canonicalisation incomplete'
assert df.payment_gateway.nunique() == 5, 'canonicalisation incomplete'

### Impossible values

Three distinct cases, and the distinction matters:

- **Non-positive amounts** — refund rows mis-ingested into the payment stream. Not payments;
  removed.
- **Internal test transactions** — absurd amounts on a sentinel customer. Removed.
- **100-year account ages** — a sentinel, not a payment defect. The *value* is nulled; the row
  stays, and notebook 03 imputes it.

What is **not** touched: the wholesale accounts with 400–900 prior orders. Those are genuine
high-value customers. Capping them away would delete a real segment.

In [ ]:
bad_amount = df.payment_amount.isna() | df.payment_amount.le(0)
test_rows  = df.payment_amount.gt(3_000_000)

df.loc[df.account_age_days_at_txn.ge(20_000) |
       df.account_age_days_at_txn.lt(0), 'account_age_days_at_txn'] = np.nan

removed = pd.DataFrame({'reason': ['non-positive / unparseable amount',
                                   'internal test transaction'],
                        'rows': [int(bad_amount.sum()), int(test_rows.sum())]})
df = df[~bad_amount & ~test_rows].copy()
print(removed.to_string(index=False))
print(f'\nrows {n_in:,} -> {len(df):,}')
print(f'account_age nulled as sentinel: {int(df.account_age_days_at_txn.isna().sum()):,}')

## 4. Clean the dimensions

Each dimension is cleaned **before** it is joined. Joining dirty dimensions and cleaning afterwards means cleaning the same value once per payment instead of once per entity, and it hides key problems behind row multiplication.

In [ ]:
od  = s3_read_csv(f'{RAW}orders.csv',        dtype=str, keep_default_na=False, na_values=[])
cu  = s3_read_csv(f'{RAW}customers.csv',     dtype=str, keep_default_na=False, na_values=[])
cd  = s3_read_csv(f'{RAW}cards.csv',         dtype=str, keep_default_na=False, na_values=[])
dv  = s3_read_csv(f'{RAW}devices.csv',       dtype=str, keep_default_na=False, na_values=[])
mr  = s3_read_csv(f'{RAW}merchants.csv',     dtype=str, keep_default_na=False, na_values=[])
ipr = s3_read_csv(f'{RAW}ip_reputation.csv', dtype=str, keep_default_na=False, na_values=[])
it  = s3_read_csv(f'{RAW}order_items.csv',   dtype=str, keep_default_na=False, na_values=[])
lg  = s3_read_csv(f'{RAW}account_logins.csv',dtype=str, keep_default_na=False, na_values=[])

# orders — order_value_inr and net_payable are dropped: exact linear functions of columns
# already present (see the leakage register in notebook 01)
for col in ['order_value', 'discount_amount', 'shipping_charge', 'item_count',
            'shipping_addr_age_hours', 'address_match_flag']:
    od[col] = to_num(od[col])
od['shipping_speed'] = canonicalise(od.shipping_speed, CANON['ship_speed'])
od['delivery_type']  = canonicalise(od.delivery_type)
for col in ['coupon_code', 'shipping_pincode', 'billing_pincode']:
    od[col] = sclean(od[col])

# customers — drop re-registered duplicate identities (no payment references them)
n_dup = int(cu.customer_id.str.endswith('_DUP').sum())
cu = cu[~cu.customer_id.str.endswith('_DUP')].copy()
cu['city']                = canonicalise(cu.city, CANON['city'])
cu['email_domain_class']  = canonicalise(cu.email_domain_class, CANON['email'])
cu['acquisition_channel'] = canonicalise(cu.acquisition_channel)
cu['kyc_level']        = to_num(cu.kyc_level).where(lambda x: x.between(0, 2))
cu['prior_return_rate'] = to_num(cu.prior_return_rate)
cu['prior_orders_12m']  = to_num(cu.prior_orders_12m)
cu['city_tier']         = to_num(cu.city_tier)
cu['home_pincode']      = sclean(cu.home_pincode)
cu['home_region_code']  = to_num(cu.home_region_code)   # authoritative column, non-null per contract
_prefix = pd.to_numeric(cu.home_pincode.str[:2], errors='coerce')
_both = cu.home_region_code.notna() & _prefix.notna()
print(f'home_region_code vs pincode prefix: agree on '
      f'{100*(cu.home_region_code[_both] == _prefix[_both]).mean():.2f}% of {int(_both.sum()):,} customers; '
      f'region nulls: {int(cu.home_region_code.isna().sum())}')
cu['signup_timestamp']  = parse_datetime(cu.signup_timestamp)

cd['bin'] = to_num(cd['bin']); cd['is_prepaid'] = to_num(cd.is_prepaid)
cd['issuer'] = sclean(cd.issuer)
cd['product_type'] = canonicalise(cd.product_type)
cd['network'] = canonicalise(cd.network, CANON['network'])
cd['issuing_country'] = sclean(cd.issuing_country)
cd['token_first_seen_timestamp'] = parse_datetime(cd.token_first_seen_timestamp)

dv['first_seen_timestamp'] = parse_datetime(dv.first_seen_timestamp)
dv['is_emulator'] = to_num(dv.is_emulator)
for col in ['device_type', 'os_family', 'browser_family']:
    dv[col] = canonicalise(dv[col], CANON[col])
    assert_levels(dv, col)
assert_levels(cd, 'network')
assert_levels(cu, 'city'); assert_levels(cu, 'email_domain_class'); assert_levels(od, 'shipping_speed')

mr['avg_ticket_size'] = to_num(mr.avg_ticket_size)
mr['trailing_chargeback_rate_bps'] = to_num(mr.trailing_chargeback_rate_bps)
mr = mr.rename(columns={'category': 'merchant_category'}).drop(columns=['merchant_name', 'city'])

ipr['is_hosting'] = to_num(ipr.is_hosting)
ipr['reputation_score'] = to_num(ipr.reputation_score)
ipr = ipr.drop(columns=['asn_name'])

n_items_raw = len(it)
it = it.drop_duplicates()
for col in ['quantity', 'unit_price', 'line_amount']:
    it[col] = to_num(it[col])

lg['login_timestamp'] = parse_datetime(lg.login_timestamp)
lg['login_risk_score'] = to_num(lg.login_risk_score)
for col in ['new_device_flag', 'unusual_location_flag', 'failed_attempt_count']:
    lg[col] = to_num(lg[col])
lg.loc[lg.failed_attempt_count.ge(100), 'failed_attempt_count'] = np.nan   # uint8 overflow

print(f'customers: {n_dup:,} duplicate identities dropped')
print(f'order_items: {n_items_raw:,} -> {len(it):,} after dedup')
print(f'logins: failed_attempt_count=255 sentinels nulled')
_ipr = df.ip_region_code
print('ip_region_code values outside the customer region set:',
      _ipr[_ipr.notna() & ~_ipr.isin(set(cu.home_region_code.dropna()))].value_counts().head(5).to_dict())
print('ip_region_code == 99 by ip_country (diagnostic only, values unchanged):')
print(pd.crosstab(_ipr.eq(99), df.ip_country.fillna('(missing)')).to_string())

## 5. Join to payment grain

The row count is asserted before and after. A silent one-to-many join is the classic way to double a dataset and then wonder why the metrics moved.

In [ ]:
before = len(df)
df = (df.merge(od.drop(columns=['customer_id', 'order_timestamp',
                                'order_value_inr', 'net_payable']), on='order_id', how='left')
        .merge(cu,  on='customer_id', how='left')
        .merge(mr,  on='merchant_id', how='left')
        .merge(cd,  on='card_token',  how='left')
        .merge(dv,  on='device_id',   how='left')
        .merge(ipr, on='ip_asn',      how='left'))
assert len(df) == before, f'join multiplied rows: {before:,} -> {len(df):,}'

basket = it.groupby('order_id').agg(n_lines=('order_item_id', 'size'),
                                    n_categories=('category', 'nunique'),
                                    max_unit_price=('unit_price', 'max'),
                                    total_qty=('quantity', 'sum'),
                                    basket_value=('line_amount', 'sum'))
df = df.merge(basket, on='order_id', how='left')
assert len(df) == before

print(f'grain preserved: {len(df):,} rows x {df.shape[1]} cols')
print(f'orders with no line items: {int(df.n_lines.isna().sum()):,} -> null, NOT zero')

### Login features — as-of join only

`merge_asof` with `allow_exact_matches=False` takes the **last login strictly before** each
payment. A plain `groupby(customer_id)` would aggregate logins that happen *after* the payment
into its features — textbook temporal leakage, and it will look like an excellent feature right
up until production, where the future does not exist.

In [ ]:
lgs = (lg[['customer_id', 'login_timestamp', 'new_device_flag', 'unusual_location_flag',
           'failed_attempt_count', 'login_risk_score']]
       .dropna(subset=['login_timestamp']).sort_values('login_timestamp'))
lgs.columns = ['customer_id', 'login_timestamp', 'last_login_new_device',
               'last_login_unusual_loc', 'last_login_failed_attempts', 'last_login_risk']

left = df[['payment_id', 'customer_id', 'payment_ts']].sort_values('payment_ts')
asof = pd.merge_asof(left, lgs, left_on='payment_ts', right_on='login_timestamp',
                     by='customer_id', allow_exact_matches=False)
asof['hours_since_last_login'] = (asof.payment_ts - asof.login_timestamp).dt.total_seconds()/3600

# Check only matched rows: a NaT comparison returns False, not null, so .fillna()
# cannot rescue it — unmatched payments would fail a test that does not apply to them.
_m = asof.login_timestamp.notna()
assert (asof.loc[_m, 'login_timestamp'] < asof.loc[_m, 'payment_ts']).all(),     'as-of join leaked forward'

df = df.merge(asof.drop(columns=['customer_id', 'payment_ts', 'login_timestamp']),
              on='payment_id', how='left')
cov = df.hours_since_last_login.notna().mean()
print(f'payments with a prior login: {int(df.hours_since_last_login.notna().sum()):,} ({100*cov:.1f}%)')
print(f'assertion passed: all {int(_m.sum()):,} matched logins strictly precede their payment')

## 6. Deterministic feature engineering

Row-wise, except the two velocity counts, which are as-of (earlier rows only; see below). None of
this can leak regardless of where the split falls.

`ip_country_missing` is deliberate: geo lookup fails more often behind hosting and VPN
infrastructure, so the missingness is **informative, not random**. Impute it away without an
indicator and you discard signal.

In [ ]:
df['card_token_age_h'] = (df.payment_ts - df.token_first_seen_timestamp).dt.total_seconds()/3600
df['device_age_h']     = (df.payment_ts - df.first_seen_timestamp).dt.total_seconds()/3600
df['account_age_days'] = df.account_age_days_at_txn.fillna(
    (df.payment_ts - df.signup_timestamp).dt.total_seconds()/86400)
df['amount_vs_merchant_ticket'] = df.payment_amount / df.avg_ticket_size

df['ip_country_mismatch'] = np.where(df.ip_country.isna(), np.nan,
                                     (df.ip_country != 'IN').astype(float))
df['ip_country_missing']  = df.ip_country.isna().astype(int)        # MAR — keep the indicator
df['ip_region_mismatch']  = np.where(df.ip_region_code.isna() | df.home_region_code.isna(),
                                     np.nan,
                                     (df.ip_region_code != df.home_region_code).astype(float))
df['issuer_foreign']      = np.where(df.issuing_country.isna(), np.nan,
                                     (df.issuing_country != 'IN').astype(float))
df['bill_ship_mismatch']  = 1 - df.address_match_flag
df['has_coupon']          = df.coupon_code.notna().astype(int)
df['is_disposable_email'] = (df.email_domain_class == 'disposable').astype(int)
df['device_orphan']       = df.device_type.isna().astype(int)       # fingerprint outage window

df['hour_of_day'] = df.payment_ts.dt.hour
df['day_of_week'] = df.payment_ts.dt.dayofweek
df['is_weekend']  = df.day_of_week.isin([5, 6]).astype(int)
df['is_night']    = df.hour_of_day.between(0, 5).astype(int)

# Point-in-time: distinct customers seen on the key AT OR BEFORE this payment (inclusive).
_full = {'device_n_customers': df.device_id.map(df.groupby('device_id').customer_id.nunique()),
         'ip_n_customers': df.ip_address.map(df.groupby('ip_address').customer_id.nunique())}  # old, record only
assert df.index.is_unique
assert df.ip_address.notna().all(), 'ip_address has nulls; ip_n_customers cannot stay int64'
df['device_n_customers'] = asof_distinct_count(df, 'device_id')
df['ip_n_customers']     = asof_distinct_count(df, 'ip_address').astype('int64')

new_cols = ['card_token_age_h', 'device_age_h', 'account_age_days',
            'amount_vs_merchant_ticket', 'ip_country_mismatch', 'ip_country_missing',
            'ip_region_mismatch', 'issuer_foreign', 'bill_ship_mismatch', 'has_coupon',
            'is_disposable_email', 'device_orphan', 'hour_of_day', 'day_of_week',
            'is_weekend', 'is_night', 'device_n_customers', 'ip_n_customers']
print(df[new_cols].describe().T[['count', 'mean', 'min', 'max']].to_string())

### Velocity features — point-in-time

`device_n_customers` and `ip_n_customers` count the distinct customers seen on the same device or IP
**at or before** each payment's `payment_ts`, including the current payment.

**Why not a full-period count.** v1 used a `groupby(...).nunique()` over the whole period, which caused two problems:

* **Look-ahead.** A January payment could see a device that was only shared in November.
* **Train/test contamination.** The statistic spanned test rows.

The pre-build check showed the stored columns equalled that full-period count on 100% of rows.

**How the as-of count works.** It uses only past rows, so it needs no split boundary, and notebook 05 re-verifies it (sentinel S3).

**Trade-off.** A lifetime count grows as history accumulates, so it drifts upward over time. A trailing-window count would be stationary. Switching is a retrain, not an edit, so it is logged as a Phase-3 candidate. At serving time the feature needs per-key first-seen state, which is a Phase-2 design decision.

In [ ]:
VELOCITY_FIX = {
    'columns': ['device_n_customers', 'ip_n_customers'],
    'method': 'as-of distinct customer count per key at or before payment_ts (inclusive); NaN where the key is null',
    'replaces': 'full-period groupby nunique (early rows saw later sharing; the statistic spanned test rows)',
    'rows_differing_from_full_period': {c: int((df[c].astype('float64').fillna(-1)
                                                != _full[c].astype('float64').fillna(-1)).sum()) for c in _full},
    'serving_note': 'needs per-key first-seen state at inference (Phase 2 decision)',
    'phase3_candidate': 'trailing-window count (stationary); a retrain, not an edit',
}
for k, v in VELOCITY_FIX.items():
    print(f'{k:32s} {v}')

## 7. Save stage 02

In [ ]:
DROP = ['payment_status', 'failure_reason', 'amount_usd', 'amount_paise', 'fee_pct']
df = df.drop(columns=[c for c in DROP if c in df.columns])
print('dropped post-decision and collinear-by-construction columns:', DROP)

assert df.payment_id.is_unique
assert df.__split.isin(['train', 'test']).all()
assert len(df) == before
assert df.payment_method.nunique() == 7

print(f'\n02_cleaned: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(s3_write_parquet(df, f'{DATA}02_cleaned.parquet'))

record = {**run_meta('02_Data_Cleaning'), 'rows_in': int(n_in), 'rows_out': int(len(df)),
          'rows_removed': removed.to_dict('records'),
          'duplicate_identities_dropped': int(n_dup),
          'order_items_deduped': int(n_items_raw - len(it)),
          'prior_login_coverage': float(cov),
          'columns_dropped': DROP, 'velocity_fix': VELOCITY_FIX}
print(s3_write_json(record, f'{REPORTS}02_run_record.json'))

## Carried into notebook 03

| Column group | Nulls | Treatment |
|---|---|---|
| `shipping_addr_age_hours` | ~4.6% | group-wise median, **fitted on train only** |
| `kyc_level`, `home_pincode`, `prior_return_rate`, `acquisition_channel` | 1.7–3.2% | mode / median + explicit `Unknown` level |
| `card_*` (issuer, network, product, bin, token age) | ~62% | **structural** — non-card payments have no card. Needs a `has_card` indicator, not imputation |
| `device_*` | ~1.4% | fingerprint outage — indicator plus `Unknown` level |
| `n_lines`, `basket_value` | 380 rows | orders with no line items — null, not zero |
| `last_login_*` | ~48% | no prior login exists — indicator, never a zero |
| `account_age_days` | sentinel-nulled | median by `city_tier`, fitted on train |

**The 62% null rate on card columns is not missingness.** It is a structural absence: a UPI
payment has no card. Imputing a median BIN across it would invent data. Notebook 03 adds
`has_card` and leaves the rest null for the tree models to branch on.

Outlier work in 03 must separate the ~800 genuine wholesale accounts (400–900 prior orders) from
true sentinels. Capping by a blanket IQR rule deletes a real customer segment.